<a href="https://colab.research.google.com/github/Renju-rl/Git-hub-DSA/blob/main/airbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [309]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score
import pickle
from sklearn.preprocessing import LabelEncoder

#load dataset

In [310]:
filepath = '/content/drive/MyDrive/DSA EXAM/data/partpdf_1772778618075_partpdf_1763620814447_airbnb.xlsx'
air = pd.read_excel(filepath)
air.head()

,Host Id,Host Since,Name,Neighbourhood,Property Type,Review Scores Rating (bin),Room Type,Zipcode,Beds,Number of Records,Number Of Reviews,Price,Review Scores Rating
0,500,2008-06-26,Gorgeous 1 BR with Private Balcony,Manhattan,Apartment,NaN,Entire home/apt,10024.0,3.0,1,0,199,NaN
1,500,2008-06-26,Trendy Times Square Loft,Manhattan,Apartment,95.0,Private room,10036.0,3.0,1,39,549,96.0
2,1039,2008-07-25,Big Greenpoint 1BD w/ Skyline View,Brooklyn,Apartment,100.0,Entire home/apt,11222.0,1.0,1,4,149,100.0
3,1783,2008-08-12,Amazing Also,Manhattan,Apartment,100.0,Entire home/apt,10004.0,1.0,1,9,250,100.0
4,2078,2008-08-15,"Colorful, quiet, & near the subway!",Brooklyn,Apartment,90.0,Private room,11201.0,1.0,1,80,90,94.0


In [311]:
air.shape

(30475, 13)

##Current Data Type of Host Since

In [312]:
air.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30475 entries, 0 to 30474
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Host Id                     30475 non-null  int64         
 1   Host Since                  30475 non-null  datetime64[ns]
 2   Name                        30475 non-null  object        
 3   Neighbourhood               30475 non-null  object        
 4   Property Type               30472 non-null  object        
 5   Review Scores Rating (bin)  22155 non-null  float64       
 6   Room Type                   30475 non-null  object        
 7   Zipcode                     30341 non-null  float64       
 8   Beds                        30390 non-null  float64       
 9   Number of Records           30475 non-null  int64         
 10  Number Of Reviews           30475 non-null  int64         
 11  Price                       30475 non-null  int64     

#Convert Host Since to DateTime Format

In [313]:
air['Host Since'] = pd.to_datetime(air['Host Since'])
air.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30475 entries, 0 to 30474
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Host Id                     30475 non-null  int64         
 1   Host Since                  30475 non-null  datetime64[ns]
 2   Name                        30475 non-null  object        
 3   Neighbourhood               30475 non-null  object        
 4   Property Type               30472 non-null  object        
 5   Review Scores Rating (bin)  22155 non-null  float64       
 6   Room Type                   30475 non-null  object        
 7   Zipcode                     30341 non-null  float64       
 8   Beds                        30390 non-null  float64       
 9   Number of Records           30475 non-null  int64         
 10  Number Of Reviews           30475 non-null  int64         
 11  Price                       30475 non-null  int64     

Possible New Features:
**Host Tenure (in days)**: Calculate how long the host has been on Airbnb
   - Formula: `(today's date - Host Since date) = days as host`


In [314]:
air['Host_Tenure_Days'] = (datetime.now() - air['Host Since']).dt.days


#Handling Missing Data

In [315]:
air.duplicated().sum()

np.int64(17)

In [316]:
air = air.drop_duplicates()

In [317]:
air.duplicated().sum()

np.int64(0)

In [318]:
air.isnull().sum()

,0
Host Id,0
Host Since,0
Name,0
Neighbourhood,0
Property Type,3
Review Scores Rating (bin),8306
Room Type,0
Zipcode,134
Beds,85
Number of Records,0


Imputation Strategy for Missing Values

The dataset contained missing values in both numerical and categorical columns. To handle these missing values, different imputation strategies were applied based on the data type of the columns.

For numerical columns, missing values were replaced using the median of each column. The median was chosen instead of the mean because numerical features, such as ratings or prices, may have skewed distributions or contain outliers. The median provides a more robust measure of central tendency and is less affected by extreme values.

For categorical columns, missing values were filled using the mode, which represents the most frequently occurring category in each column. This method preserves the most common category and helps maintain the distribution of categorical variables.

In [319]:
num_col = ['Review Scores Rating','Review Scores Rating (bin)','Zipcode','Beds']
cat_col = ['Property Type']

In [320]:
for col in num_col:
  air[col] = air[col].fillna(air[col].median())

for cols in cat_col:
  air[cols] = air[cols].fillna(air[cols].mode()[0])

In [321]:
air.isnull().sum()

,0
Host Id,0
Host Since,0
Name,0
Neighbourhood,0
Property Type,0
Review Scores Rating (bin),0
Room Type,0
Zipcode,0
Beds,0
Number of Records,0


In [322]:
air.head()

,Host Id,Host Since,Name,Neighbourhood,Property Type,Review Scores Rating (bin),Room Type,Zipcode,Beds,Number of Records,Number Of Reviews,Price,Review Scores Rating,Host_Tenure_Days
0,500,2008-06-26,Gorgeous 1 BR with Private Balcony,Manhattan,Apartment,90.0,Entire home/apt,10024.0,3.0,1,0,199,94.0,6462
1,500,2008-06-26,Trendy Times Square Loft,Manhattan,Apartment,95.0,Private room,10036.0,3.0,1,39,549,96.0,6462
2,1039,2008-07-25,Big Greenpoint 1BD w/ Skyline View,Brooklyn,Apartment,100.0,Entire home/apt,11222.0,1.0,1,4,149,100.0,6433
3,1783,2008-08-12,Amazing Also,Manhattan,Apartment,100.0,Entire home/apt,10004.0,1.0,1,9,250,100.0,6415
4,2078,2008-08-15,"Colorful, quiet, & near the subway!",Brooklyn,Apartment,90.0,Private room,11201.0,1.0,1,80,90,94.0,6412


#Advanced Feature Engineering

###Interaction Feature

In [323]:
air["Neighbourhood_RoomType"] = air["Neighbourhood "] + "_" + air["Room Type"]

###Encoding Categorical Columns

In [324]:
le_neighbourhood = LabelEncoder()
le_room_type = LabelEncoder()
le_property_type = LabelEncoder()
le_neighbourhood_roomtype = LabelEncoder()

air['Neighbourhood '] = le_neighbourhood.fit_transform(air['Neighbourhood '])
air['Room Type'] = le_room_type.fit_transform(air['Room Type'])
air['Property Type'] = le_property_type.fit_transform(air['Property Type'])
air['Neighbourhood_RoomType'] = le_neighbourhood_roomtype.fit_transform(air['Neighbourhood_RoomType'])

In [325]:
air.head()

,Host Id,Host Since,Name,Neighbourhood,Property Type,Review Scores Rating (bin),Room Type,Zipcode,Beds,Number of Records,Number Of Reviews,Price,Review Scores Rating,Host_Tenure_Days,Neighbourhood_RoomType
0,500,2008-06-26,Gorgeous 1 BR with Private Balcony,2,0,90.0,0,10024.0,3.0,1,0,199,94.0,6462,6
1,500,2008-06-26,Trendy Times Square Loft,2,0,95.0,1,10036.0,3.0,1,39,549,96.0,6462,7
2,1039,2008-07-25,Big Greenpoint 1BD w/ Skyline View,1,0,100.0,0,11222.0,1.0,1,4,149,100.0,6433,3
3,1783,2008-08-12,Amazing Also,2,0,100.0,0,10004.0,1.0,1,9,250,100.0,6415,6
4,2078,2008-08-15,"Colorful, quiet, & near the subway!",1,0,90.0,1,11201.0,1.0,1,80,90,94.0,6412,4


###Select Features without new feature

In [326]:
features = [
    "Neighbourhood ",
    "Property Type",
    "Review Scores Rating (bin)",
    "Room Type",
    "Zipcode",
    "Beds",
    "Number of Records",
    "Number Of Reviews",
    "Review Scores Rating",
    "Host_Tenure_Days"
]

X1 = air[features]
y = air["Price"]

###features with new feature

In [327]:
features1 = [
    "Neighbourhood ",
    "Property Type",
    "Review Scores Rating (bin)",
    "Room Type",
    "Zipcode",
    "Beds",
    "Number of Records",
    "Number Of Reviews",
    "Review Scores Rating",
    "Host_Tenure_Days",
    "Neighbourhood_RoomType"

]

X2 = air[features1]


###Train-Test Split

In [328]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y, test_size=0.2, random_state=42)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)

###Train Random Forest Model without new feature

In [329]:
rf1 = RandomForestRegressor(random_state=42)

rf1.fit(X1_train, y1_train)

pred1 = rf1.predict(X1_test)

rmse1 = np.sqrt(mean_squared_error(y1_test, pred1))
r2 = r2_score(y1_test, pred1)


print("RMSE without interaction feature:", rmse1)
print("RMSE without interaction feature:", r2)

RMSE without interaction feature: 205.85288639701966
RMSE without interaction feature: 0.11258195461118592


######Train Random Forest Model with new feature

In [330]:
rf2 = RandomForestRegressor(random_state=42)

rf2.fit(X2_train, y2_train)

pred2 = rf2.predict(X2_test)

rmse2 = np.sqrt(mean_squared_error(y2_test, pred2))
r2_1 = r2_score(y2_test, pred2)


print("RMSE without interaction feature:", rmse2)
print("RMSE without interaction feature:", r2_1)

RMSE without interaction feature: 205.62317182611307
RMSE without interaction feature: 0.11456141788445084


In [331]:
rmse1 = 205.85
rmse2 =  205.62
r2 = 0.1125
r2_1 =0.1125

comparison_df = pd.DataFrame({
    'Metric': ['RMSE', 'R² Score'],
    'Baseline': [f'${rmse1:.2f}', f'{r2:.4f}'],
    'Enhanced': [f'${rmse2:.2f}', f'{r2_1:.4f}']
})

print(comparison_df)

     Metric Baseline Enhanced
0      RMSE  $205.85  $205.62
1  R² Score   0.1125   0.1125


Adding the interaction feature slightly improved the model’s performance, reducing RMSE from 205.85 to 205.62 and increasing R² from 0.113 to 0.115.

The interaction feature captures more nuanced information because it allows the model to understand how the combination of Neighbourhood and Room Type jointly influences the target variable. For instance, a Private room in a high-demand Neighbourhood may have a different pricing pattern than the same room type in a less popular area. Such interactions are often not captured when features are considered independently.

### Interpretation of Final Model's RMSE

Your final model, which includes the `Neighbourhood_RoomType` interaction feature, achieved a Root Mean Squared Error (RMSE) of approximately **$205.62**.

**What does this mean in practical terms for an Airbnb host?**

RMSE represents the typical magnitude of the errors made by your model in predicting prices. In simpler terms, if your model predicts an Airbnb listing's price, on average, that prediction will be about **$205.62 off** from the actual price. This 'off' can be either higher or lower than the true price.

For example:
*   If the model predicts a listing should cost $150, the actual price could realistically be anywhere from around $150 - $205.62 = $-55.62 (which is not possible for price, implying that actual price will be greater than 0) to $150 + $205.62 = $355.62.
*   If a host sets a price of $300, the model's prediction for that listing might typically fall between $300 - $205.62 = $94.38 and $300 + $205.62 = $505.62.

**In essence:** An RMSE of $205.62 indicates that while the model captures some trends, its individual price predictions for Airbnb listings still have a notable average deviation of around $205.62 from the true market price. For a host, this means the model provides a general idea of pricing, but it's not highly precise for individual listings.

In [332]:
print("\nSaving trained model with pickle...")

# Save the model
with open('airbnb_price_model.pkl', 'wb') as f:
    pickle.dump(rf2, f)
print("✓ Model saved as 'airbnb_price_model.pkl'")

# Also save the label encoders
with open('le_room_type.pkl', 'wb') as f:
    pickle.dump(le_room_type, f)
print("✓ Room Type encoder saved as 'le_room_type.pkl'")

with open('le_property_type.pkl', 'wb') as f:
    pickle.dump(le_property_type, f)
print("✓ Property Type encoder saved as 'le_property_type.pkl'")

with open('le_neighbourhood.pkl', 'wb') as f:
    pickle.dump(le_neighbourhood, f)
print("✓ Neighbourhood encoder saved as 'le_neighbourhood.pkl'")

# 'le_review_bin' was not explicitly created or used in the previous steps.
# If 'Review Scores Rating (bin)' was meant to be encoded and saved, a separate encoder
# instance should have been created and fitted for it. For now, this line is removed.
# with open('le_review_bin.pkl', 'wb') as f:
#     pickle.dump(le_review_bin, f)
# print("✓ Review Bin encoder saved as 'le_review_bin.pkl'")

with open('le_neighbourhood_roomtype.pkl', 'wb') as f:
    pickle.dump(le_neighbourhood_roomtype, f)
print("✓ Neighbourhood_RoomType encoder saved as 'le_neighbourhood_roomtype.pkl'")

print("\n" + "=" * 80)
print("ALL FILES SAVED SUCCESSFULLY!")
print("=" * 80)


Saving trained model with pickle...
✓ Model saved as 'airbnb_price_model.pkl'
✓ Room Type encoder saved as 'le_room_type.pkl'
✓ Property Type encoder saved as 'le_property_type.pkl'
✓ Neighbourhood encoder saved as 'le_neighbourhood.pkl'
✓ Neighbourhood_RoomType encoder saved as 'le_neighbourhood_roomtype.pkl'

ALL FILES SAVED SUCCESSFULLY!
